In [1]:
#%pip install fitz docling pytesseract pdf2image easyocr opencv-python pandas tqdm pillow
#%pip install frontend
#%pip install tools
#%pip install pyMuPDF
from PIL import Image
import os
import pandas as pd
import glob
from tqdm import tqdm
import fitz

from pathlib import Path

from docling.backend.docling_parse_backend import DoclingParseDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    EasyOcrOptions,
    OcrMacOptions,
    PdfPipelineOptions,
    RapidOcrOptions,
    TesseractCliOcrOptions,
    TesseractOcrOptions,
)
from docling.document_converter import DocumentConverter, PdfFormatOption

In [2]:
import torch
print(torch.cuda.is_available())  # Deve retornar True se a GPU estiver disponível
print(torch.cuda.current_device())  # Deve retornar o índice do dispositivo CUDA atual
print(torch.cuda.get_device_name(torch.cuda.current_device()))  # Deve retornar o nome do dispositivo CUDA atual

True
0
NVIDIA GeForce RTX 3060


In [20]:
source = glob.glob("C:/Estudos e Projetos/(2024) Dados LAI MP/separado_doc_tjsp/unificador_documentos/pasta_teste_juncao_docs/sentença/*.pdf")[0:1]

In [14]:
def extrai_texto(path_to_pdf_file):
    input_doc = Path(fr"{path_to_pdf_file}")

    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = True
    # habilita a extração de tabelas
    pipeline_options.do_table_structure = True
    pipeline_options.table_structure_options.do_cell_matching = True
    pipeline_options.ocr_options.lang = ["pt-br"]


    # Any of the OCR options can be used:EasyOcrOptions, TesseractOcrOptions, TesseractCliOcrOptions, OcrMacOptions(Mac only), RapidOcrOptions
    # ocr_options = EasyOcrOptions(force_full_page_ocr=True)
    # ocr_options = TesseractOcrOptions(force_full_page_ocr=True)
    # ocr_options = OcrMacOptions(force_full_page_ocr=True)
    # ocr_options = RapidOcrOptions(force_full_page_ocr=True)
    ocr_options = TesseractCliOcrOptions(force_full_page_ocr=True)
    pipeline_options.ocr_options = ocr_options

    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options,
            )
        }
    )

    doc = converter.convert(input_doc).document
    md = doc.export_to_markdown()
    return md

In [15]:
# salva uma página
def converte_imagem(source):
    """
    Input: Caminho para um documento PDF;
    Output: Retorna uma imagem em preto e branco com 300 DPI.
    """
    # armaena os dados md
    print(source)
    lista_md = []

    # extrai metadados
    numero_tj = source.split("\\")[1].split("_")[0]
    tipo_doc = source.split('\\')[1].split("_",1)[1].split(".")[0]

    # Abrir o documento PDF
    pdf_document = fitz.open(source)

    # Salvar cada página como uma imagem em preto e branco com 300 DPI
    for i in tqdm(range(len(pdf_document))):
        md_blocks = []
        page = pdf_document.load_page(i)
        pix = page.get_pixmap(dpi=300)
        image = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
        bw_image = image.convert("L")  # Converter para preto e branco

        # extrai texto e converte em markdown
        md = extrai_texto(image)
        md_blocks.append(md)

        # juntar os blocos de markdown em um único texto
        md_text = "\n".join(md_blocks)

        # salvar md_text numa lista 
        lista_md.append([numero_tj, tipo_doc, md_text])

        df = pd.DataFrame(lista_md, columns=["numero_tj","tipo_doc","texto_md"])

        return df

        # image_path = os.path.join(output_dir, f"page_{i + 1}.png")
        # bw_image.save(image_path, dpi=(300, 300))
        # print(f"Página {i + 1} salva como {image_path}")

In [151]:
#### teste

In [22]:
def extrai_texto_de_imagem(image, roi=None):

    # Se uma região de interesse (ROI) for fornecida, recortar a imagem
    if roi:
        image = image.crop(roi)

    # Converter a imagem para preto e branco
    bw_image = image.convert("L")

    # Salvar a imagem temporariamente para processamento
    temp_image_path = Path(r"C:/Estudos e Projetos/2024-LLM-Agronomo-Virtual/1 - criação_base/pasta_docs_tmp/temp_image.png")
    bw_image.save(temp_image_path)

    # Configurar as opções do pipeline de OCR
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = True
    pipeline_options.do_table_structure = True
    pipeline_options.table_structure_options.do_cell_matching = True
    pipeline_options.cuda_device = 0  # Usar a GPU 0

    # Configurar as opções do OCR
    ocr_options = TesseractCliOcrOptions(force_full_page_ocr=True)
    ocr_options.device = 'cuda'  # Configurar para usar CUDA
    ocr_options.lang = ["pt-br"]  # Definir o idioma para português
    pipeline_options.ocr_options = ocr_options

    # Criar o conversor de documentos
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options,
            )
        }
    )

    # Converter a imagem temporária para texto
    doc = converter.convert(temp_image_path).document
    md = doc.export_to_markdown()

    # Remover a imagem temporária
    temp_image_path.unlink()

    return md

In [23]:
for item in source:    
    """
    Iterates over a list of source items, extracts metadata, opens PDF documents, and processes each page as a black and white image with 300 DPI.
    Args:
        source (list): List of file paths to PDF documents.
    Variables:
        lista_md (list): List to store metadata and extracted text.
        numero_tj (str): Extracted metadata representing the document number.
        tipo_doc (str): Extracted metadata representing the document type.
        pdf_document (fitz.Document): PDF document object.
        md_blocks (list): List to store markdown text blocks for each page.
        page (fitz.Page): Page object of the PDF document.
        pix (fitz.Pixmap): Pixmap object representing the page image.
        image (PIL.Image): Image object created from the pixmap.
        bw_image (PIL.Image): Black and white version of the image.
    Note:
        The code for extracting text and converting it to markdown is commented out.
        The code for appending the markdown text to the list and creating a DataFrame is also commented out.
    """
    # Definir a região de interesse (ROI) como uma tupla (left, upper, right, lower)
    roi = (100, 100, 400, 400)
    
    print(source)
    lista_md = []

    # extrai metadados
    numero_tj = item.split("\\")[1].split("_")[0]
    tipo_doc = item.split('\\')[1].split("_",1)[1].split(".")[0]

    # Abrir o documento PDF
    pdf_document = fitz.open(item.replace("\\","/"))

    # Salvar cada página como uma imagem em preto e branco com 300 DPI
    for i in tqdm(range(len(pdf_document))):
        md_blocks = []
        page = pdf_document.load_page(i)
        pix = page.get_pixmap(dpi=300)
        image = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)

        # # extrai texto e converte em markdown
        md = extrai_texto_de_imagem(image,roi=roi)
        print(md)
        md_blocks.append(md)

        # juntar os blocos de markdown em um único texto
        md_text = "\n".join(md_blocks)

        # salvar md_text numa lista 
        lista_md.append([numero_tj, tipo_doc, md_text])

        df = pd.DataFrame(lista_md, columns=["numero_tj","tipo_doc","texto_md"])

        df


['C:/Estudos e Projetos/(2024) Dados LAI MP/separado_doc_tjsp/unificador_documentos/pasta_teste_juncao_docs/sentença\\0000049-51.2018.8.26.0603_sentença_pags_210-218.pdf']


AttributeError: module 'fitz' has no attribute 'open'

In [29]:
%pip uninstall pyMuPDF


^C
Note: you may need to restart the kernel to use updated packages.


In [136]:
if __name__ == "__main__":
    for item in source:
        converte_imagem(item)


C:/Estudos e Projetos/(2024) Dados LAI MP/separado_doc_tjsp/unificador_documentos/pasta_teste_juncao_docs/sentença\0000049-51.2018.8.26.0603_sentença_pags_210-218.pdf


  0%|          | 0/9 [00:00<?, ?it/s]


OSError: [Errno 22] Invalid argument: '<PIL.Image.Image image mode=RGB size=2484x3509 at 0x1DE29367200>'